# Lawgic Document Inference Pipeline

This notebook implements an end-to-end document-level inference pipeline for the Lawgic dual-head Legal-BERT classifier. It processes raw `.txt` Terms of Service documents by:

1. **Reading & normalizing** the document text (UTF-8, line-ending normalization)
2. **Segmenting** the document into paragraphs via `\n\n` delimiters
3. **Defensively chunking** oversized paragraphs using NLTK sentence tokenization to stay within the model's trained context window
4. **Running batched inference** through the fine-tuned `LawgicDualHeadModel` (topic presence + consumer harm)
5. **Producing a structured per-clause report** as a Pandas DataFrame

The model predicts:
- **Topic presence** (44-class multi-label, sigmoid activation, configurable threshold)
- **Consumer harm level** (3-class multi-class: Harmful / Neutral / Fair, softmax activation)

See: `docs/lawgic_dual_head_architecture.md` for the full model architecture documentation.

## Imports & NLTK Setup

In [1]:
import torch
import torch.nn as nn
import json
import nltk
import pandas as pd
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

# Download NLTK sentence tokenizer data (required for sub-chunking fallback)
nltk.download('punkt_tab', quiet=True)

print("All imports loaded successfully.")

All imports loaded successfully.


## Configuration Constants

All tunable pipeline parameters are centralized here. Adjust `TOPIC_THRESHOLD` to control
topic detection sensitivity. The `TOKEN_THRESHOLD` of 200 is derived from the training
`max_length` of 256 minus special tokens and a WordPiece inflation buffer.

In [2]:
# ── Path to the saved dual-head model directory ──
MODEL_DIR = Path("../../saved_models/lawgic_classifier_legal-bert_v3").resolve()

# ── Inference parameters (must match training conditions) ──
MAX_LENGTH = 256            # Tokenizer max_length — matches training max_length
TOKEN_THRESHOLD = 200       # Defensive sub-chunking threshold (accounts for [CLS]/[SEP] + WordPiece inflation)

# ── Classification parameters ──
TOPIC_THRESHOLD = 0.5       # Sigmoid decision boundary for topic presence (configurable)
BATCH_SIZE = 16             # Inference batch size for parallel tensor processing

# ── Model dimensions ──
NUM_TOPICS = 44             # Number of Lawgic topic classes
NUM_HARM_CLASSES = 3        # Number of harm level classes
HARM_CLASS_NAMES = {0: "Harmful", 1: "Neutral", 2: "Fair"}

# ── Paragraph filtering ──
MIN_CLAUSE_LENGTH = 15      # Minimum character length for a paragraph to be considered substantive

print(f"Model directory: {MODEL_DIR}")
print(f"Token threshold: {TOKEN_THRESHOLD} | Max length: {MAX_LENGTH}")
print(f"Topic threshold: {TOPIC_THRESHOLD} | Batch size: {BATCH_SIZE}")

Model directory: /Users/riki/Coding Projects/Thesis/lawgic/saved_models/lawgic_classifier_legal-bert_v3
Token threshold: 200 | Max length: 256
Topic threshold: 0.5 | Batch size: 16


## Hardware Detection

The pipeline dynamically detects the best available hardware accelerator:
1. **CUDA** (NVIDIA GPU) — preferred for maximum throughput
2. **MPS** (Apple Silicon GPU) — used on macOS with M-series chips
3. **CPU** — fallback when no GPU is available

In [3]:
# ── Device detection: CUDA > MPS > CPU ──
if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple Silicon (MPS)"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print(f"Using device: {device} ({device_name})")

Using device: mps (Apple Silicon (MPS))


## Model Architecture: LawgicDualHeadModel

The dual-head model wraps `AutoModel` (Legal-BERT encoder) with two independent classification heads:

- **Topic Head** — `nn.Linear(768, 44)` → multi-label topic presence (sigmoid at inference)
- **Harm Head** — `nn.Linear(768, 3)` → multi-class consumer harm (softmax at inference)

Both heads operate on the shared `[CLS]` pooler output from the encoder.

See `docs/lawgic_dual_head_architecture.md` for the complete architectural specification.

In [4]:
class LawgicDualHeadModel(nn.Module):
    """Dual-head Legal-BERT model for simultaneous topic and harm classification."""
    
    def __init__(self, model_name: str, num_topics: int = NUM_TOPICS, num_harm_classes: int = NUM_HARM_CLASSES):
        super().__init__()
        # Load the pre-trained Legal-BERT encoder
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size  # 768 for legal-bert-base
        
        # Independent classification heads on the shared [CLS] embedding
        self.topic_head = nn.Linear(hidden_size, num_topics)    # 768 -> 44
        self.harm_head = nn.Linear(hidden_size, num_harm_classes)  # 768 -> 3
        
        self.num_topics = num_topics
        self.num_harm_classes = num_harm_classes

    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        # Build encoder kwargs, optionally including token_type_ids
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        
        # Forward pass through the shared encoder
        outputs = self.encoder(**encoder_kwargs)
        cls_embedding = outputs.pooler_output  # (batch_size, 768)
        
        # Route through both heads independently
        topic_logits = self.topic_head(cls_embedding)  # (batch_size, 44)
        harm_logits = self.harm_head(cls_embedding)    # (batch_size, 3)
        
        return topic_logits, harm_logits

## Load Tokenizer, Model & Topic Map

Load the fine-tuned model weights, tokenizer, and the 44-topic label mapping from the saved model directory.

In [5]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

print("Loading model...")
# Instantiate the dual-head model using the saved encoder config
model = LawgicDualHeadModel(str(MODEL_DIR))

# Load the full state dict (encoder + both classification heads)
state_dict_path = MODEL_DIR / "model_state_dict.pt"
if state_dict_path.exists():
    model.load_state_dict(
        torch.load(state_dict_path, map_location=device, weights_only=True)
    )
else:
    # Fallback: load head weights individually if full state dict is missing
    topic_weights = torch.load(MODEL_DIR / "topic_head_weights.pt", map_location=device, weights_only=True)
    harm_weights = torch.load(MODEL_DIR / "harm_head_weights.pt", map_location=device, weights_only=True)
    model.topic_head.load_state_dict(topic_weights)
    model.harm_head.load_state_dict(harm_weights)

# Move model to the detected device and set to evaluation mode
model = model.to(device)
model.eval()

# Load the 44-topic label mapping for decoding predictions
topics_json_path = MODEL_DIR / "lawgic_topics_44.json"
with topics_json_path.open("r", encoding="utf-8") as f:
    topics_data = json.load(f)

id2label = {t["classifier_id"]: t["topic_id"] for t in topics_data}
id2name = {t["classifier_id"]: t["name"] for t in topics_data}

print(f"Model loaded on {device} | {len(id2label)} topics | {NUM_HARM_CLASSES} harm classes")

Loading tokenizer...
Loading model...
Model loaded on mps | 44 topics | 3 harm classes


---

## Step A: Document Reading & Normalization

The first pipeline stage ingests a raw `.txt` file and normalizes it for consistent downstream processing:

- **UTF-8 encoding** with `errors="replace"` to handle malformed byte sequences gracefully
- **Line-ending normalization**: converts Windows-style `\r\n` to standard `\n`
- **Whitespace trimming**: strips leading/trailing whitespace from the full document

In [6]:
def read_document(file_path: Path) -> str:
    """
    Read a .txt file and return its normalized text content.
    
    Handles UTF-8 encoding with graceful fallback for malformed bytes.
    Normalizes all line endings to \n for consistent downstream splitting.
    """
    file_path = Path(file_path)  # Ensure Path object
    
    if not file_path.exists():
        raise FileNotFoundError(f"Document not found: {file_path}")
    if not file_path.suffix == ".txt":
        raise ValueError(f"Expected .txt file, got: {file_path.suffix}")
    
    # Read with UTF-8 encoding; replace malformed bytes instead of crashing
    raw_text = file_path.read_text(encoding="utf-8", errors="replace")
    
    # Normalize Windows line endings (\r\n) to standard Unix (\n)
    normalized = raw_text.replace("\r\n", "\n")
    
    # Strip leading/trailing whitespace from the entire document
    normalized = normalized.strip()
    
    print(f"Document read: {file_path.name} | {len(normalized):,} characters | {normalized.count(chr(10)):,} lines")
    return normalized

---

## Step B: Paragraph Segmentation

Legal documents use double-newline (`\n\n`) boundaries to separate distinct provisions,
clauses, or itemized rights. This stage splits the normalized text on these boundaries.

Paragraphs shorter than `MIN_CLAUSE_LENGTH` characters (e.g., section headers like
`"DEFINITIONS."` or date stamps) are **retained in the output** with `skipped=True` so the
full document structure is preserved, but they are excluded from inference.

In [7]:
def segment_paragraphs(text: str, min_length: int = MIN_CLAUSE_LENGTH) -> list:
    """
    Split normalized text into paragraph-level segments using \n\n delimiters.
    
    Returns a list of dicts, each containing:
      - paragraph_id: sequential index
      - text: the stripped paragraph text
      - skipped: True if the paragraph is too short for meaningful inference
    
    Skipped paragraphs (headers, whitespace, date stamps) are preserved in the
    output for document structure traceability but excluded from inference.
    """
    # Split on double-newline boundaries (standard legal document paragraph separators)
    raw_fragments = text.split("\n\n")
    
    paragraphs = []
    paragraph_id = 0
    
    for fragment in raw_fragments:
        stripped = fragment.strip()
        
        # Skip completely empty fragments (artifacts of multiple consecutive newlines)
        if not stripped:
            continue
        
        # Determine if this fragment is substantive enough for inference
        is_skipped = len(stripped) < min_length
        
        paragraphs.append({
            "paragraph_id": paragraph_id,
            "text": stripped,
            "skipped": is_skipped
        })
        paragraph_id += 1
    
    # Report segmentation statistics
    total = len(paragraphs)
    skipped_count = sum(1 for p in paragraphs if p["skipped"])
    substantive_count = total - skipped_count
    print(f"Segmented: {total} paragraphs | {substantive_count} substantive | {skipped_count} skipped")
    
    return paragraphs

---

## Step C: Defensive Token Budgeting & Sub-Chunking

The model was trained with `max_length=256`. To ensure inference conditions match training:

- **Token threshold = 200**: Accounts for 2 special tokens (`[CLS]`, `[SEP]`) and
  WordPiece tokenization inflation (~15–25% over raw word count)
- **Pass-through**: Paragraphs under 200 tokens are queued directly for inference
- **Sub-chunking fallback**: Paragraphs exceeding 200 tokens are split into sentences
  using `nltk.sent_tokenize()`, then greedily recombined into sub-chunks that stay
  under the threshold. No sentence is ever split mid-phrase.

Training data statistics: median=38, p95=137, p99=268, max=3,579 tokens.
The overwhelming majority of real-world clauses pass through without sub-chunking.

In [8]:
def estimate_tokens(text: str, tokenizer) -> int:
    """
    Return the exact token count for a text string using the model's tokenizer.
    
    Uses add_special_tokens=False to count only content tokens,
    since special tokens ([CLS], [SEP]) are accounted for in the threshold budget.
    """
    return len(tokenizer.encode(text, add_special_tokens=False))

In [9]:
def chunk_paragraph(paragraph: str, tokenizer, threshold: int = TOKEN_THRESHOLD) -> list:
    """
    Split a paragraph into inference-safe chunks that respect the token budget.
    
    Logic:
      1. If the paragraph fits within the threshold -> return as a single chunk
      2. Otherwise -> split into sentences via NLTK, then greedily recombine
         sentences into sub-chunks that stay under the threshold
      3. If a single sentence exceeds the threshold, it becomes its own chunk
         (the tokenizer will truncate it at MAX_LENGTH during inference)
    
    Returns a list of chunk strings.
    """
    token_count = estimate_tokens(paragraph, tokenizer)
    
    # Pass-through: paragraph fits within the token budget
    if token_count <= threshold:
        return [paragraph]
    
    # Fallback: sentence-level splitting for oversized paragraphs
    sentences = nltk.sent_tokenize(paragraph)
    
    chunks = []              # Accumulated safe chunks
    current_chunk = []       # Sentences being accumulated for the current chunk
    current_tokens = 0       # Running token count for the current chunk
    
    for sentence in sentences:
        sentence_tokens = estimate_tokens(sentence, tokenizer)
        
        # Check if adding this sentence would exceed the threshold
        if current_tokens + sentence_tokens > threshold and current_chunk:
            # Flush the current chunk before it overflows
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_tokens = 0
        
        # Append the sentence to the current chunk
        current_chunk.append(sentence)
        current_tokens += sentence_tokens
    
    # Flush any remaining sentences in the buffer
    if current_chunk:
        chunks.append(" ".join(current_chunk))
    
    return chunks

In [10]:
def prepare_chunks(paragraphs: list, tokenizer, threshold: int = TOKEN_THRESHOLD) -> list:
    """
    Process all substantive (non-skipped) paragraphs into inference-ready chunks.
    
    Each chunk is a dict containing:
      - chunk_id: global sequential index across all chunks
      - paragraph_id: the source paragraph's index
      - text: the chunk text
      - estimated_tokens: exact token count
      - is_subchunk: True if the paragraph was split into multiple chunks
    """
    all_chunks = []
    chunk_id = 0
    subchunked_paragraphs = 0  # Counter for paragraphs that required splitting
    
    for para in paragraphs:
        # Skip paragraphs flagged as non-substantive
        if para["skipped"]:
            continue
        
        # Apply defensive chunking to this paragraph
        sub_chunks = chunk_paragraph(para["text"], tokenizer, threshold)
        is_subchunked = len(sub_chunks) > 1
        
        if is_subchunked:
            subchunked_paragraphs += 1
        
        for chunk_text in sub_chunks:
            all_chunks.append({
                "chunk_id": chunk_id,
                "paragraph_id": para["paragraph_id"],
                "text": chunk_text,
                "estimated_tokens": estimate_tokens(chunk_text, tokenizer),
                "is_subchunk": is_subchunked
            })
            chunk_id += 1
    
    # Report chunking statistics
    print(f"Prepared: {len(all_chunks)} inference chunks from "
          f"{len([p for p in paragraphs if not p['skipped']])} substantive paragraphs")
    print(f"Sub-chunked paragraphs: {subchunked_paragraphs} "
          f"(exceeded {threshold}-token threshold)")
    
    if all_chunks:
        token_counts = [c["estimated_tokens"] for c in all_chunks]
        print(f"Token distribution: min={min(token_counts)}, "
              f"median={sorted(token_counts)[len(token_counts)//2]}, "
              f"max={max(token_counts)}")
    
    return all_chunks

---

## Step D: Batched Tensor Inference

Chunks are collated into batches of `BATCH_SIZE` for efficient GPU/MPS tensor processing:

- **Tokenization**: `padding=True`, `truncation=True`, `max_length=256`
- **Non-gradient context**: `torch.no_grad()` disables gradient computation for memory efficiency
- **Activation routing**:
  - Topic head → `sigmoid` (independent probability per topic, multi-label)
  - Harm head → `softmax` (mutually exclusive probability distribution, multi-class)

In [11]:
def run_batch_inference(
    chunks: list,
    model: nn.Module,
    tokenizer,
    device: torch.device,
    batch_size: int = BATCH_SIZE,
    topic_threshold: float = TOPIC_THRESHOLD
) -> list:
    """
    Run dual-head inference on all chunks in batches.
    
    For each chunk, produces:
      - topic_probabilities: dict mapping all 44 topic names to their sigmoid probability
      - predicted_topics: list of topic names exceeding the threshold
      - harm_probabilities: dict mapping harm class names to softmax probabilities
      - predicted_harm_class: the dominant harm class name
      - harm_confidence: the softmax probability of the predicted class
    """
    results = []
    total_batches = (len(chunks) + batch_size - 1) // batch_size  # Ceiling division
    
    for batch_idx in range(0, len(chunks), batch_size):
        batch_chunks = chunks[batch_idx : batch_idx + batch_size]
        batch_texts = [c["text"] for c in batch_chunks]
        current_batch = batch_idx // batch_size + 1
        
        # Tokenize the batch with padding and truncation to MAX_LENGTH
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )
        
        # Move all tensors to the active device
        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)
        token_type_ids = encoded.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device)
        
        # Forward pass with gradient tracking disabled for memory efficiency
        with torch.no_grad():
            topic_logits, harm_logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        
        # Apply activation functions to raw logits
        topic_probs = torch.sigmoid(topic_logits).cpu()  # (batch, 44) — independent per-topic
        harm_probs = torch.softmax(harm_logits, dim=-1).cpu()  # (batch, 3) — mutually exclusive
        harm_preds = torch.argmax(harm_logits, dim=-1).cpu()   # (batch,) — predicted class index
        
        # Unpack each sample in the batch into the results list
        for i, chunk in enumerate(batch_chunks):
            # Build topic probability dict and filter by threshold
            sample_topic_probs = topic_probs[i].tolist()
            topic_prob_dict = {}
            predicted_topics = []
            
            for topic_idx, prob in enumerate(sample_topic_probs):
                topic_name = id2name.get(topic_idx, f"Topic_{topic_idx}")
                topic_prob_dict[topic_name] = round(prob, 4)
                if prob >= topic_threshold:
                    predicted_topics.append(topic_name)
            
            # Build harm probability dict
            sample_harm_probs = harm_probs[i].tolist()
            harm_prob_dict = {
                HARM_CLASS_NAMES[idx]: round(p, 4)
                for idx, p in enumerate(sample_harm_probs)
            }
            
            # Get the predicted harm class and its confidence
            harm_class_idx = harm_preds[i].item()
            predicted_harm = HARM_CLASS_NAMES.get(harm_class_idx, "Unknown")
            harm_confidence = sample_harm_probs[harm_class_idx]
            
            results.append({
                "chunk_id": chunk["chunk_id"],
                "paragraph_id": chunk["paragraph_id"],
                "text": chunk["text"],
                "estimated_tokens": chunk["estimated_tokens"],
                "is_subchunk": chunk["is_subchunk"],
                "topic_probabilities": topic_prob_dict,
                "predicted_topics": predicted_topics,
                "harm_probabilities": harm_prob_dict,
                "predicted_harm_class": predicted_harm,
                "harm_confidence": round(harm_confidence, 4)
            })
        
        print(f"  Batch {current_batch}/{total_batches}: processed {len(batch_chunks)} chunks")
    
    print(f"Inference complete: {len(results)} chunks processed.")
    return results

---

## Step E: Structured Report Assembly

The inference results are assembled into a Pandas DataFrame that merges:
- **Inference rows**: chunks with topic predictions, harm classifications, and confidence scores
- **Skipped rows**: section headers and short fragments with `skipped=True` and `NaN` prediction columns

This preserves the full document structure while clearly distinguishing substantive analysis from structural markers.

In [12]:
def build_results_dataframe(results: list, paragraphs: list) -> pd.DataFrame:
    """
    Combine inference results with skipped paragraphs into a unified DataFrame.
    
    Inference rows have full prediction data. Skipped rows have NaN for all
    prediction columns but retain their text and paragraph_id for traceability.
    """
    rows = []
    
    # Add inference result rows
    for r in results:
        topics_str = ", ".join(r["predicted_topics"]) if r["predicted_topics"] else "(none)"
        rows.append({
            "chunk_id": r["chunk_id"],
            "paragraph_id": r["paragraph_id"],
            "text_preview": r["text"][:100] + ("..." if len(r["text"]) > 100 else ""),
            "predicted_topics": topics_str,
            "harm_class": r["predicted_harm_class"],
            "harm_confidence": r["harm_confidence"],
            "is_subchunk": r["is_subchunk"],
            "skipped": False,
            "estimated_tokens": r["estimated_tokens"]
        })
    
    # Add skipped paragraph rows
    for p in paragraphs:
        if p["skipped"]:
            rows.append({
                "chunk_id": None,
                "paragraph_id": p["paragraph_id"],
                "text_preview": p["text"][:100] + ("..." if len(p["text"]) > 100 else ""),
                "predicted_topics": None,
                "harm_class": None,
                "harm_confidence": None,
                "is_subchunk": False,
                "skipped": True,
                "estimated_tokens": None
            })
    
    # Build DataFrame and sort by paragraph_id for document-order readability
    df = pd.DataFrame(rows)
    df = df.sort_values(by=["paragraph_id", "chunk_id"], na_position="first").reset_index(drop=True)
    
    return df

In [13]:
def print_summary_statistics(df: pd.DataFrame, results: list):
    """
    Print a summary of the pipeline run including paragraph counts,
    harm distribution, and most frequently predicted topics.
    """
    print("=" * 70)
    print("PIPELINE SUMMARY")
    print("=" * 70)
    
    # Paragraph and chunk counts
    total_paragraphs = len(df["paragraph_id"].unique())
    skipped_count = df["skipped"].sum()
    inference_chunks = len(results)
    subchunk_count = df[df["is_subchunk"] == True]["paragraph_id"].nunique()
    
    print(f"\nTotal paragraphs:           {total_paragraphs}")
    print(f"Skipped (headers/short):    {skipped_count}")
    print(f"Inference chunks:           {inference_chunks}")
    print(f"Paragraphs sub-chunked:     {subchunk_count}")
    
    # Harm class distribution (inference rows only)
    inference_df = df[~df["skipped"]]
    if not inference_df.empty:
        print(f"\n--- Harm Class Distribution ---")
        harm_counts = inference_df["harm_class"].value_counts()
        for harm_class, count in harm_counts.items():
            pct = count / len(inference_df) * 100
            print(f"  {harm_class}: {count} ({pct:.1f}%)")
    
    # Top predicted topics (aggregated across all chunks)
    if results:
        topic_counter = {}
        for r in results:
            for topic in r["predicted_topics"]:
                topic_counter[topic] = topic_counter.get(topic, 0) + 1
        
        if topic_counter:
            print(f"\n--- Top 10 Predicted Topics ---")
            sorted_topics = sorted(topic_counter.items(), key=lambda x: x[1], reverse=True)[:10]
            for topic_name, count in sorted_topics:
                print(f"  {topic_name}: {count} chunks")
        else:
            print(f"\nNo topics predicted above threshold ({TOPIC_THRESHOLD}).")
    
    print("\n" + "=" * 70)

---

## Pipeline Execution: Sample ToS

Run the full pipeline on `data/sample_tos/sample_tos.txt` — a synthetic ToS document
designed to exercise all pipeline stages including sub-chunking of oversized paragraphs.

In [14]:
# ══════════════════════════════════════════════════════════════════════
# PIPELINE RUN 1: Synthetic Sample ToS
# ══════════════════════════════════════════════════════════════════════

TOS_FILE_SAMPLE = Path("../../data/sample_tos/sample_tos.txt").resolve()
print(f"Target: {TOS_FILE_SAMPLE}")
print()

# Step A: Read & normalize the document
raw_text_sample = read_document(TOS_FILE_SAMPLE)
print()

# Step B: Segment into paragraphs (with skip flagging)
paragraphs_sample = segment_paragraphs(raw_text_sample)
print()

# Step C: Defensive chunking (sub-chunk oversized paragraphs)
chunks_sample = prepare_chunks(paragraphs_sample, tokenizer, TOKEN_THRESHOLD)
print()

# Step D: Batched dual-head inference
results_sample = run_batch_inference(chunks_sample, model, tokenizer, device, BATCH_SIZE, TOPIC_THRESHOLD)
print()

# Step E: Build structured report
df_sample = build_results_dataframe(results_sample, paragraphs_sample)

Target: /Users/riki/Coding Projects/Thesis/lawgic/data/sample_tos/sample_tos.txt

Document read: sample_tos.txt | 7,429 characters | 75 lines

Segmented: 38 paragraphs | 37 substantive | 1 skipped

Prepared: 39 inference chunks from 37 substantive paragraphs
Sub-chunked paragraphs: 2 (exceeded 200-token threshold)
Token distribution: min=3, median=24, max=198

  Batch 1/3: processed 16 chunks
  Batch 2/3: processed 16 chunks
  Batch 3/3: processed 7 chunks
Inference complete: 39 chunks processed.



In [15]:
# Display the full results DataFrame for the sample ToS
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', None)
display(df_sample)

print()
print_summary_statistics(df_sample, results_sample)

,chunk_id,paragraph_id,text_preview,predicted_topics,harm_class,harm_confidence,is_subchunk,skipped,estimated_tokens
0,0.0,0,"Terms of Service\nLast Updated: March 15, 2025",Contract Changes,Neutral,1.0000,False,False,10.0
1,1.0,1,"Welcome to ExampleApp (""Service,"" ""we,"" ""us,"" or ""our""). By accessing or using our Service, you ...","Contract Changes, Notice of Changes, Contract Formed Through Use, Privacy Policy Incorporation, ...",Neutral,0.8323,False,False,63.0
2,2.0,2,1. Acceptance of Terms,Notice of Changes,Harmful,0.9999,False,False,5.0
3,3.0,3,"By creating an account, accessing, or using the Service in any way, you acknowledge that you hav...","Choice of Law, Choice of Forum, Notice of Changes, Privacy Policy Incorporation",Neutral,0.9993,False,False,67.0
4,4.0,4,2. Changes to Terms,"Contract Changes, Notice of Changes",Harmful,0.9999,False,False,5.0
5,5.0,5,We reserve the right to modify these Terms at any time. We will notify you of any material chang...,Notice of Changes,Neutral,0.8853,False,False,73.0
6,6.0,6,3. Account Registration,"Recommender System Transparency, Interpretation Clause, Transparency",Neutral,0.9979,False,False,4.0
7,7.0,7,You must provide accurate and complete information when creating an account. You are responsible...,Security,Neutral,0.9999,False,False,34.0
8,8.0,8,4. Privacy Policy,"Recommender System Transparency, Interpretation Clause, Transparency",Fair,0.9557,False,False,4.0
9,9.0,9,Our collection and use of personal information in connection with the Service is described in ou...,"Choice of Law, Choice of Forum, Limitation of Liability, Liability Cap, Warranty Disclaimer, Ind...",Neutral,1.0000,False,False,28.0



PIPELINE SUMMARY

Total paragraphs:           38
Skipped (headers/short):    1
Inference chunks:           39
Paragraphs sub-chunked:     2

--- Harm Class Distribution ---
  Neutral: 19 (48.7%)
  Harmful: 12 (30.8%)
  Fair: 8 (20.5%)

--- Top 10 Predicted Topics ---
  Contract Formed Through Use: 9 chunks
  Notice of Changes: 7 chunks
  Service Governance: 7 chunks
  Recommender System Transparency: 7 chunks
  Complaint Handling System: 7 chunks
  Privacy Policy Incorporation: 6 chunks
  Interpretation Clause: 6 chunks
  Transparency: 6 chunks
  Discretionary Interpretation: 6 chunks
  Severability: 6 chunks



---

## Pipeline Execution: Apollo.io Terms of Service

Run the pipeline on a real-world Terms of Service document: Apollo.io's ToS
(`data/new_tos/apollo_io.txt`). This is a 188-line, ~56KB document with several
extremely long paragraphs — including a full-page warranty disclaimer and multi-paragraph
arbitration sections — that will naturally exercise the sub-chunking fallback path.

In [16]:
# ══════════════════════════════════════════════════════════════════════
# PIPELINE RUN 2: Apollo.io Terms of Service (Real-World Document)
# ══════════════════════════════════════════════════════════════════════

TOS_FILE_APOLLO = Path("../../data/new_tos/apollo_io.txt").resolve()
print(f"Target: {TOS_FILE_APOLLO}")
print()

# Step A: Read & normalize the document
raw_text_apollo = read_document(TOS_FILE_APOLLO)
print()

# Step B: Segment into paragraphs (with skip flagging)
paragraphs_apollo = segment_paragraphs(raw_text_apollo)
print()

# Step C: Defensive chunking (sub-chunk oversized paragraphs)
chunks_apollo = prepare_chunks(paragraphs_apollo, tokenizer, TOKEN_THRESHOLD)
print()

# Step D: Batched dual-head inference
results_apollo = run_batch_inference(chunks_apollo, model, tokenizer, device, BATCH_SIZE, TOPIC_THRESHOLD)
print()

# Step E: Build structured report
df_apollo = build_results_dataframe(results_apollo, paragraphs_apollo)

Token indices sequence length is longer than the specified maximum sequence length for this model (5919 > 512). Running this sequence through the model will result in indexing errors


Target: /Users/riki/Coding Projects/Thesis/lawgic/data/new_tos/apollo_io.txt

Document read: apollo_io.txt | 56,500 characters | 187 lines

Segmented: 20 paragraphs | 20 substantive | 0 skipped

Prepared: 75 inference chunks from 20 substantive paragraphs
Sub-chunked paragraphs: 6 (exceeded 200-token threshold)
Token distribution: min=5, median=173, max=255

  Batch 1/5: processed 16 chunks
  Batch 2/5: processed 16 chunks
  Batch 3/5: processed 16 chunks
  Batch 4/5: processed 16 chunks
  Batch 5/5: processed 11 chunks
Inference complete: 75 chunks processed.



In [17]:
# Display the full results DataFrame for the Apollo.io ToS
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.max_rows', None)
display(df_apollo)

print()
print_summary_statistics(df_apollo, results_apollo)

,chunk_id,paragraph_id,text_preview,predicted_topics,harm_class,harm_confidence,is_subchunk,skipped,estimated_tokens
0,0,0,"Terms of Service\nLast Updated: February 5, 2026",Contract Changes,Neutral,1.0000,False,False,10
1,1,1,These Terms of Service (“Terms of Service” or “Terms”) are a legally binding agreement between y...,"Choice of Law, Choice of Forum, Mandatory Arbitration, Class Action Waiver, Contract Changes, No...",Neutral,0.9997,False,False,79
2,2,2,You accept and agree to these Terms of Service by:,"Liability Cap, Contract Changes, Notice of Changes, Contract Formed Through Use, Privacy Policy ...",Neutral,0.9876,False,False,11
3,3,3,"Accessing or using the Service;\nClicking to accept these Terms of Service, or\nAccepting these ...","Liability Cap, Contract Changes, Notice of Changes, Contract Formed Through Use, Privacy Policy ...",Neutral,0.8570,False,False,60
4,4,4,Important: Please note Sections 6 and 12 of these Terms which include important information rega...,"Mandatory Arbitration, Class Action Waiver",Harmful,0.9998,False,False,48
5,5,5,AUTOMATIC RENEWAL NOTICE: Your subscription will automatically renew for additional periods of t...,Payments,Neutral,0.9999,False,False,62
6,6,6,We may modify these Terms of Service (except for Section 7) in our sole discretion by posting up...,Notice of Changes,Harmful,0.9997,False,False,64
7,7,7,DEFINITIONS. “Apollo DPA” means the Data Processing Addendum found at: https://www.apollo.io/dpa...,"Recommender System Transparency, Interpretation Clause, Transparency",Neutral,0.9999,True,False,143
8,8,7,“Business Contact Information” means information about a natural person in a professional contex...,"Privacy Policy Incorporation, Personal Data",Harmful,0.9996,True,False,154
9,9,7,"“Customer Third-Party Systems” means any third-party products, systems, applications, or service...","Third Parties, Recommender System Transparency, Interpretation Clause, Transparency",Neutral,1.0000,True,False,177



PIPELINE SUMMARY

Total paragraphs:           20
Skipped (headers/short):    0
Inference chunks:           75
Paragraphs sub-chunked:     6

--- Harm Class Distribution ---
  Neutral: 40 (53.3%)
  Harmful: 30 (40.0%)
  Fair: 5 (6.7%)

--- Top 10 Predicted Topics ---
  Mandatory Arbitration: 16 chunks
  Class Action Waiver: 16 chunks
  Third Parties: 13 chunks
  Contract Formed Through Use: 12 chunks
  Contract Changes: 10 chunks
  Service Governance: 10 chunks
  Notice of Changes: 9 chunks
  Complaint Handling System: 7 chunks
  Discretionary Interpretation: 6 chunks
  Account Termination: 6 chunks

